# Alexandria v2: better sentences, 50 levels, a closer look at the evaluator

Builds on the models from the first notebook (`artifacts/` on Drive) and writes the new ones to
`artifacts_v2/`, so the old ones keep working until you switch the backend over.

| Step | What | Runtime | Rough cost |
|---|---|---|---|
| 1 | Diagnose the current evaluator | any (CPU ok) | ~0 |
| 2 | Difficulty v2: 50 fixed levels, spoken-Hindi word ranks (reuses v1 counts) | CPU | ~0 |
| 3 | Naturalness model: an LLM rates 20k sentences, a small model learns from it | L4 | ~10-15 CU |
| 4 | Generator v2: filtered everyday data, continues from your v1 adapter | L4 | ~25-40 CU |
| 5 | Evaluator v2: learner mistakes + paraphrases, continues from v1; label sheet for your team | L4 | ~5-10 CU |
| 6 | Fill the easy levels with generated sentences | L4 | ~5-10 CU |

**Runtime:** Runtime → Change runtime type → **L4 GPU** (bfloat16, 24 GB; faster and more stable than a T4).
Steps 1-2 need no GPU: you can run them on a CPU runtime first to save units.

Every step writes to Drive and can be re-run; training resumes from its last checkpoint.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT = "/content/drive/MyDrive/alexandria"
A1 = f"{ROOT}/artifacts"      # v1 models (from the first notebook)
A = f"{ROOT}/artifacts_v2"    # v2 models: point the backend here when you are happy with them
D = f"{ROOT}/data_v2"
R = f"{ROOT}/reports"
for p in (A, D, R): os.makedirs(p, exist_ok=True)
os.environ["ALEXANDRIA_CACHE"] = f"{ROOT}/cache"

# Must be the preset your v1 generator was trained with, so its adapter can be continued.
PRESET = "qwen3-4b"

In [ ]:
BRANCH = "main"
!test -d /content/alexandria || git clone -b {BRANCH} https://github.com/SIGMobile-UIUC/alexandria.git /content/alexandria
%cd /content/alexandria
!git pull -q
!pip install -q -e "models[train,serve]"
# Colab ships an old torchao that peft refuses to import; we don't use it.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print("No HF_TOKEN secret; gated datasets (BPCC, Samanantar) will be skipped:", e)

import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU runtime (fine for steps 1-2)")

## 1. How does the current evaluator do? (no training)
Scores 300 sentence pairs against every kind of learner mistake. Look for rows marked `<-- check`:
for example a flipped negation that still scores 0.6, or a harmless contraction that scores 0.7.

In [ ]:
!python -m alexandria_models.evaluator.diagnose --evaluator_dir {A1}/evaluator --out {R}/evaluator_v1.json

## 2. Difficulty v2: 50 levels anchored to what a learner knows
Level 1 = two or three of the ~200 most common words with है (यह घर है।). Word ranks blend web text
with spoken Hindi (film subtitles, Tatoeba, everyday conversation), so news words count as rare.
Reuses the v1 word counts, so this takes minutes instead of an hour.

In [ ]:
!python -m alexandria_models.difficulty.build --output_dir {A}/difficulty --reuse_from {A1}/difficulty

In [ ]:
import json
s = json.load(open(f"{A}/difficulty/scorer.json"))
for lv in s["level_summaries"]:
    ex = lv["examples"][0] if lv["examples"] else "(no corpus sentences: step 6 fills these)"
    print(f"level {lv['level']:2d}  n={lv['count']:6d}  {ex}")

## 3. Naturalness model
**3a.** Collect candidate sentences with weak labels (everyday corpora good, news/fragments bad).
**3b.** An open LLM (Qwen3-8B) rates 20,000 of them 1-5 for "would a teacher use this?". About 15-30 min on an L4;
re-run the cell after a disconnect and it continues where it stopped. On a T4 add `--load_in_4bit`.
**3c.** Train a small MuRIL classifier on those ratings (~10 min); it is what filters everything else.

In [ ]:
!python -m alexandria_models.naturalness.prepare_data --output_dir {D}/naturalness

In [ ]:
!python -m alexandria_models.naturalness.label_llm --input {D}/naturalness/to_judge.jsonl --output {D}/naturalness/judged.jsonl

In [ ]:
# What did the judge think? Lowest and highest rated sentences:
import json
rows = sorted((json.loads(l) for l in open(f"{D}/naturalness/judged.jsonl")), key=lambda r: r["rating"])
for r in rows[:8] + rows[-8:]:
    print(f"{r['rating']:.1f}  [{r['source']}]  {r['hi']}")

In [ ]:
!python -m alexandria_models.naturalness.train --data_dir {D}/naturalness --output_dir {A}/naturalness --resume

In [ ]:
from alexandria_models.naturalness.score import NaturalnessModel
nat = NaturalnessModel.load(f"{A}/naturalness")
tests = ["मैं दूध पीता हूँ।", "क्या आप चाय लेंगे?", "मेरे पिता का अधिक कुछ नहीं कहना है।",
         "महोदय, कृपया, सर।", "खाद्यान्न उत्पादन में विकास", "इस दौरान मोदी ने भारत और रूस के बीच संबंधों को और मजबूत बनाने पर जोर दिया है।"]
for t, sc in zip(tests, nat.score_batch(tests)):
    print(f"{sc:.2f}  {t}")

## 4. Generator v2
**4a.** Data: complete everyday sentences only (rules + naturalness ≥ 0.5), 50 levels, up to 2000 per level.
Scoring the candidates with the naturalness model uses the GPU (~5-10 min).
**4b.** Training continues from your v1 adapter (`--init_adapter`), so it starts from what it already learned.
About 2-3 h on an L4. If Colab disconnects, re-run the cell: `--resume` picks up from the last checkpoint.

In [ ]:
!python -m alexandria_models.generator.prepare_data --difficulty_dir {A}/difficulty --output_dir {D}/generator \
    --naturalness_dir {A}/naturalness

In [ ]:
import json, collections
cfg = json.load(open(f"{D}/generator/data_config.json"))
print("examples:", cfg["train_examples"], "| per level:", cfg["per_level"])
bank = [json.loads(l) for l in open(f"{D}/generator/sentence_bank.jsonl")]
by = collections.defaultdict(list)
for r in bank: by[r["level"]].append(r)
for lv in (1, 3, 5, 10, 20, 30, 40, 50):
    for r in by.get(lv, [])[:2]:
        print(f"L{lv:2d} [{r['source']}] {r['hi']}  —  {r['en']}")

In [ ]:
!python -m alexandria_models.generator.train --data_dir {D}/generator --output_dir {A}/generator \
    --preset {PRESET} --init_adapter {A1}/generator/adapter --resume

## 5. Evaluator v2
New training data adds realistic learner mistakes (dropped ने, English word order, missing articles and
auxiliaries) and correct answers that are worded differently (paraphrases, contractions, हूं/हूँ), which
must score high. Training continues from the v1 model. Then the diagnosis is re-run so you can compare.

In [ ]:
!python -m alexandria_models.evaluator.prepare_data --output_dir {D}/evaluator
!python -m alexandria_models.evaluator.train --data_dir {D}/evaluator --output_dir {A}/evaluator \
    --base_model {A1}/evaluator/model --epochs 1 --resume

In [ ]:
!python -m alexandria_models.evaluator.diagnose --evaluator_dir {A}/evaluator --out {R}/evaluator_v2.json

**Your team's test set.** This writes `label_sheet.csv` (200 answers) and `RUBRIC.txt` to Drive under `reports/`.
Open the CSV in Google Sheets, fill the `score` column 0-4 using the rubric, download it as CSV back to the
same path, then run the next cell. You can also replace answers with real learner attempts (romanized is fine).

In [ ]:
!python -m alexandria_models.evaluator.label_sheet make --pairs {D}/generator/sentence_bank.jsonl --out {R}/label_sheet.csv

In [ ]:
# After the sheet is scored: how do v1 and v2 compare with your team's judgement?
for ev in ("artifacts", "artifacts_v2"):
    print("==", ev)
    !python -m alexandria_models.evaluator.label_sheet score --sheet {R}/label_sheet.csv \
        --evaluator_dir {ROOT}/{ev}/evaluator --transliterator_dir {A1}/transliterator \
        --difficulty_dir {A}/difficulty --export {D}/evaluator/test_learner.jsonl --out {R}/label_sheet_{ev}.json

## 6. Fill the easy levels with generated sentences
Real corpora have few very simple sentences. For every level with fewer than 200 bank sentences, the generator
writes the rest; each one must pass the learner rules, the naturalness model, the level check (±2) and the
evaluator's translation check.

In [ ]:
!cp -rn {A1}/transliterator {A}/
!python -m alexandria_models.generator.generate --generator_dir {A}/generator --difficulty_dir {A}/difficulty \
    --evaluator_dir {A}/evaluator --naturalness_dir {A}/naturalness \
    --bank {A}/generator/sentence_bank.jsonl --fill_to 200 --out {A}/generator/generated_bank.jsonl

## 7. Try everything

In [ ]:
from alexandria_models.inference import AlexandriaModels, COMPONENTS
m = AlexandriaModels.load(A, components=COMPONENTS)
print(m.status())
for lv in (1, 2, 5, 10, 25, 40):
    for s in m.sentences(level=lv, n=2, mode="bank"):
        print(f"L{lv:2d} [{s['source']}] {s['hindi']} — {s['english']}")
print(m.sentences(level=3, n=3, mode="generate"))
print(m.evaluate("मैं दूध पीता हूँ।", "I drink milk"))
print(m.evaluate("I drink milk.", "main doodh peeta hoon"))
print(m.evaluate("मैं दूध नहीं पीता हूँ।", "I drink milk"))

**Switching the backend:** point it at `artifacts_v2` (e.g. `ALEXANDRIA_ARTIFACTS=/path/to/artifacts_v2`).
Download it without the training leftovers with the cell below.

In [ ]:
!cd {ROOT} && zip -qr /content/artifacts_v2.zip artifacts_v2 -x "*/checkpoints/*" "*/adapter/*" && ls -lh /content/artifacts_v2.zip
from google.colab import files
files.download("/content/artifacts_v2.zip")